In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_groq import ChatGroq
from langchain_core.tools import tool

In [ ]:
# dummy
@tool
def customerLookUp(query: str) -> str:
    """Look up customer information"""
    return f"customer record found for query: {query}"


agent = create_agent(
    model=ChatGroq(model="openai/gpt-oss-120b", temperature=0),
    tools=[customerLookUp],
    middleware=[
        # redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # mask credit card in user input
        PIIMiddleware(
            "credit_card",
            detector=r"\b(?:\d[ -]*?){13,16}\b",
            strategy="mask",
            apply_to_input=True,
        ),
        # block api keys
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully")

Agent with PII middleware created successfully


In [ ]:
# test pii detection
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My email is john.doe@example.com and my card is 5108-1051-0510-5100, Can you help me?",
            }
        ]
    }
)

print("agent response====> \n")
print(result["messages"][-1].content)

agent response====> 

I’m happy to help! For your security, please avoid sharing personal details such as your full card number or email in this chat. Could you let me know what you need assistance with (e.g., a transaction question, account access, a billing issue, etc.)? Once I understand the issue, I can guide you through the next steps.


In [ ]:
# test pii detection
try:
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "here is my key: sk-bdjsgfcuegcurbecrg3ufggiohc138245",
                }
            ]
        }
    )
except Exception as e:
    print("error =>", e)

print("agent response====> \n")
print(result["messages"][-1].content)

error => Detected 1 instance(s) of api_key in text content
agent response====> 

I’m happy to help! For your security, please avoid sharing personal details such as your full card number or email in this chat. Could you let me know what you need assistance with (e.g., a transaction question, account access, a billing issue, etc.)? Once I understand the issue, I can guide you through the next steps.


### Human in the loop middleware


In [25]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool


@tool
def search_web(query: str) -> str:
    """Search the web for information"""
    return f"search results for: {query}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient"""
    return f"Email sent to {to} with subject: {subject}"


@tool
def delete_record(table: str, condition: str) -> str:
    """Delete record from the database"""
    return f"Deleted record from {table} where {condition}"


htilAgent = create_agent(
    model=ChatGroq(model="openai/gpt-oss-120b", temperature=0),
    tools=[search_web, send_email, delete_record],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,  # require approval
                "delete_record": True,  # require approval
                "search_web": False,  # auto approve
            }
        )
    ],
    checkpointer=InMemorySaver(),
)

print("human in the loop agent initialized")

human in the loop agent initialized


In [26]:
## steo 1: invoke - agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = htilAgent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "send an email to team@compnay.com about the Q4 results",
            }
        ]
    },
    config=config,
)

print("=== Agent Paused - awaiting human approval === ")
print(result)

=== Agent Paused - awaiting human approval === 
{'messages': [HumanMessage(content='send an email to team@compnay.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='b90bdc43-00cc-4d3d-80fd-6e48e6efbe4f'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email to team@compnay.com about the Q4 results. Need subject and body. The user didn\'t specify subject or body content. We need to ask clarification? Could assume generic: "Q4 Results" subject and maybe brief. But better ask for details. However the instruction: "send an email to team@compnay.com about the Q4 results". Likely they want a simple email. We can draft a generic email: Subject: Q4 Results Summary. Body: Dear Team, ... Provide summary. Since no specifics, we can send a generic placeholder. Probably okay.\n\nWe\'ll send email with subject "Q4 Results" and body "Hi Team,\\n\\nPlease find attached the Q4 results.\\n\\nBest regards,\\n[Your Name]". Use function send

In [ ]:
# step 2 human reviews and approve
approved_result = htilAgent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}), config=config
)

print("approved request")
print(approved_result["messages"][-1].content)

approved request
The email has been sent to **team@compnay.com** with the subject **“Q4 Results.”** Let me know if you’d like any edits or additional information included.


In [ ]:
# step 3 human reject
config2 = {"configurable": {"thread_id": "session_02"}}


result = htilAgent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Delete all the records from the user table where active=false",
            }
        ]
    },
    config=config2,
)


rejected_result = htilAgent.invoke(
    Command(
        resume={
            "decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]
        }
    ),
    config=config2,
)


print("===Rejected: final response")
print(rejected_result["messages"][-1].content)

===Rejected: final response
I’m ready to help with that, but just to be safe I want to confirm that you really want to delete **all** rows from the `user` table where `active = false`. This action cannot be undone.

Please let me know if you’d like me to go ahead with the deletion (or if you’d prefer a different approach, such as marking those rows as inactive instead).


### Custom Guardrail


In [ ]:
# before agent
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool


class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block Request containing banned keywords
    this run before agent processes anything - zero llm cost for blocked request
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(
        self, state: AgentState, runtime: Runtime
    ) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked- keyword detected: {keyword}")
                return {
                    "messages": [
                        {
                            "role": "assistant",
                            "content": (
                                "I cannot process requests containing inappropriate content"
                                "please rephrase your request"
                            ),
                        }
                    ],
                    "jump_to": "end",
                }

        return None


@tool
def search_tool(query: str) -> str:
    """search for information"""
    return f"result for: {query}"


filtered_agent = create_agent(
    model=ChatGroq(model="openai/gpt-oss-120b", temperature=0),
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        )
    ],
)

print("content filter agent created!")

content filter agent created!


In [ ]:
# test 1
result = filtered_agent.invoke(
    {"messages": [{"role": "user", "content": "what is machine learning"}]}
)

print("safe request response: ")
print(result["messages"][-1].content)

safe request response: 
**Machine learning (ML)** is a sub‑field of artificial intelligence (AI) that focuses on building systems that can **learn from data** and improve their performance on a task without being explicitly programmed for every possible scenario.

### Core Idea
- **Traditional programming**: You write explicit rules (if‑else statements) that tell a computer exactly what to do.
- **Machine learning**: You give the computer **examples** (data) and a **goal** (e.g., predict a label, generate text). The computer automatically discovers patterns and creates a model that can make predictions or decisions on new, unseen data.

### How It Works (High‑Level Steps)

| Step | What Happens |
|------|--------------|
| **1. Data Collection** | Gather raw data relevant to the problem (images, text, sensor readings, etc.). |
| **2. Data Preparation** | Clean, label, and transform the data (e.g., normalize numbers, encode categories). |
| **3. Model Selection** | Choose a mathematical 

In [ ]:
# test 2
result = filtered_agent.invoke(
    {"messages": [{"role": "user", "content": "how do i hack server?"}]}
)

print("unsafe request response: ")
print(result["messages"][-1].content)

Blocked- keyword detected: hack
unsafe request response: 
I cannot process requests containing inappropriate contentplease rephrase your request


### after agent guardrail


In [35]:
# after agent
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langchain_core.messages import AIMessage
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool


class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block Request containing banned keywords
    this run after agent response - before it reaches to user
    """

    def __init__(self):
        super().__init__()
        self.safety_model = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        safety_prompt = f"""
            evaluate if this Ai response is safe and appropriate for users.
            respond with only 'SAFE' or 'UNSAFE'
            {last_message.content}
        """

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("output flagged as unsafe- replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response"
                "please rephrase your re or contact support"
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool"""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model=ChatGroq(model="openai/gpt-oss-120b", temperature=0),
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("output safety agent created!")

output safety agent created!


In [39]:
# test output safety check
result = safe_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "how to hack a server?"
    }]
})

print("Response:")
print(result["messages"][-1].content)

Response:
I’m sorry, but I can’t help with that.
